# Read sav data

## read_spss

In [ ]:
import pandas as pd
data_locat = "Pew-Research-Center-Global-Attitudes-Spring-2025-Public/Pew Research Center Global Attitudes Spring 2025 Dataset.sav"
df = pd.read_spss(data_locat)  # pandas wraps pyreadstat internally
df.head()

## pyreadstat.read_sav

In [ ]:
import pyreadstat

df2, meta = pyreadstat.read_sav(data_locat)
print(df2.head())
print(meta.column_names[1:10])      # variable names
print(meta.column_labels[1:10])     # variable labels
type(meta.value_labels) # Data dictionary, hard to present easily
# Contains value labels (e.g. 1="Male", 2="Female")

## Data dictionary  

In [ ]:
data_dict = pd.DataFrame({
    "variable": meta.column_names,
    "label": meta.column_labels
})

# Code below affects display of all dataframes
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_colwidth", None)
# display(data_dict)

# If want to reset to normal...
# pd.reset_option("all")

# Just this data frame...
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(data_dict)

# The with keyword here is used as a context manager. 
# It applies a temporary setting for the duration of the indented block. 
# Then automatically reverts it afterwards.


## A widget for filtering variables

In [ ]:
import ipywidgets as widgets
from IPython.display import display

data_dict = pd.DataFrame({
    "variable": meta.column_names,
    "label": meta.column_labels,
    "values": [meta.variable_value_labels.get(var, {}) for var in meta.column_names]
})

search = widgets.Text(placeholder="Search variables or labels...")
pos_from = widgets.IntText(value=-1, description="From:")
pos_to = widgets.IntText(value=-1, description="To:")
reset_btn = widgets.Button(description="Reset", button_style="warning")
output = widgets.Output()

def show_variable(var_name):
    row = data_dict.loc[data_dict["variable"] == var_name].iloc[0]
    print(f"Variable : {row['variable']}")
    print(f"Label    : {row['label']}")
    print()

    if row["values"]:
        val_df = pd.DataFrame(
            row["values"].items(),
            columns=["value", "value_label"]
        ).sort_values("value")
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(val_df.reset_index(drop=True))
    else:
        print("No value labels defined.")

def show_full_table():
    with pd.option_context("display.max_rows", 20, "display.max_colwidth", None):
        display(data_dict)

def filter_table(change):
    query = change["new"].lower()
    with output:
        output.clear_output()
        if not query:
            show_full_table()
            return
        mask = (
            data_dict["variable"].str.lower().str.contains(query) |
            data_dict["label"].str.lower().str.contains(query)
        )
        matches = data_dict[mask]
        if len(matches) == 1:
            show_variable(matches["variable"].values[0])
        else:
            with pd.option_context("display.max_rows", 20, "display.max_colwidth", None):
                display(matches)

def filter_by_position(change):
    start = pos_from.value
    end = pos_to.value
    with output:
        output.clear_output()
        if start < 0 and end < 0:
            show_full_table()
            return
        if start < 0:
            start = end
        if end < 0:
            end = start
        if start > end:
            print("'From' must be less than or equal to 'To'.")
            return
        if start >= len(data_dict) or end >= len(data_dict):
            print(f"Position out of range. Must be between 0 and {len(data_dict)-1}.")
            return
        search.unobserve(filter_table, names="value")
        search.value = ""
        search.observe(filter_table, names="value")
        subset = data_dict.iloc[start:end+1]
        if len(subset) == 1:
            show_variable(subset["variable"].values[0])
        else:
            with pd.option_context("display.max_rows", 20, "display.max_colwidth", None):
                display(subset)

def on_reset(b):
    search.unobserve(filter_table, names="value")
    pos_from.unobserve(filter_by_position, names="value")
    pos_to.unobserve(filter_by_position, names="value")
    search.value = ""
    pos_from.value = -1
    pos_to.value = -1
    search.observe(filter_table, names="value")
    pos_from.observe(filter_by_position, names="value")
    pos_to.observe(filter_by_position, names="value")
    with output:
        output.clear_output()
        show_full_table()

search.observe(filter_table, names="value")
pos_from.observe(filter_by_position, names="value")
pos_to.observe(filter_by_position, names="value")
reset_btn.on_click(on_reset)

with output:
    show_full_table()

display(widgets.HBox([search, pos_from, pos_to, reset_btn]), output)

# Descriptive statistics

## A frequency count

In [ ]:
df2["econ_sit"].value_counts()

### Converting to a data frame

In [ ]:
vc = df2["econ_sit"].value_counts().reset_index()
vc.columns = ["code", "count"]
vc["label"] = vc["code"].map(meta.variable_value_labels["econ_sit"])
vc = vc[["code", "label", "count"]].sort_values("code")
print(vc, type(vc))

### Adjusting to a function

In [ ]:
def jfreq(df, var): 
    vc = df[var].value_counts().reset_index()
    vc.columns = ["code", "count"]
    vc["label"] = vc["code"].map(meta.variable_value_labels[var])
    vc = vc[["code", "label", "count"]].sort_values("code")
    return vc

jfreq(df2, var = "econ_sit")

In [ ]:
# Note when run on df, from pd.read_spss, this works less well as data has already had codes removed
jfreq(df, var = "econ_sit")

In [ ]:
jfreq(df2, var = "satisfied_democracy")

In [ ]:
jfreq(df2, var = "officials_trait_ethical")

In [ ]:
jfreq(df2, var = "officials_trait_understand")

# Relevant variables


econ_sit
satisfied_democracy
polsys_satisfied
moral_state
moral_affair
moral_homosexuality	
moral_abortion

In [ ]:
jfreq(df2, var = "moral_affair")

# K-medoids

In [ ]:
df3 = df2.loc[:, df2.columns.str.contains('moral')]

In [ ]:
df3

In [ ]:
for x in df3.columns:
    print(x)
    display(jfreq(df3, var = x))

In [ ]:
df4 = df3.drop(columns=['moral_state'])
df4.columns

In [ ]:
print("Before:", len(df4))
df_filtered = df4[~df4[df4.columns].isin([8, 9]).any(axis=1)]
print("After:", len(df_filtered))
print("Dropped:", len(df4) - len(df_filtered))

In [ ]:
df_filtered

In [ ]:
import sklearn; print(sklearn.__version__)

In [ ]:
from sklearn.cluster import KMedoids

kmed = KMedoids(n_clusters=3, metric='hamming', random_state=42)
kmed.fit(df_filtered.astype(str))

df_filtered['cluster'] = kmed.labels_